Sentiment Analysis - Trained from Dataset

In [6]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load the JSON dataset from file
with open('nlp_authentication_dataset.json', 'r') as file:
    data = json.load(file)

# Step 2: Extract texts and map sentiments to 1 (positive), 0 (negative), or skip neutral
texts = []
sentiments = []

for entry in data:
    sentiment = entry['sentiment'].lower()  # Convert to lowercase for consistency
    if sentiment == 'positive':
        sentiments.append(1)
    elif sentiment == 'negative':
        sentiments.append(0)
    else:
        continue  # Skip neutral or unrecognized sentiments
    texts.append(entry['raw_text'])  # Only append text if sentiment is valid

# Ensure there are valid texts and sentiments
if len(texts) == 0 or len(sentiments) == 0:
    raise ValueError("No valid data available for training after filtering.")

# Step 3: Tokenization and padding
tokenizer = Tokenizer(num_words=10000)  # Increase vocab size for better representation
tokenizer.fit_on_texts(texts)
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, padding='post', maxlen=200)  # Increase maxlen for longer texts

# Step 4: Convert sentiments to numpy arrays
y = np.array(sentiments)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Build a more complex LSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=128, input_length=200),  # Larger embedding size
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),  # More units, return sequences for deeper layers
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),  # Additional LSTM layer
    tf.keras.layers.Dropout(0.5),  # Dropout for regularization
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Step 7: Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Step 8: Use Early Stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Step 9: Train the model with more epochs and early stopping
model.fit(np.array(X_train), np.array(y_train), epochs=20, batch_size=32, validation_data=(np.array(X_test), np.array(y_test)), callbacks=[early_stopping])

# Step 10: Evaluate the model
loss, accuracy = model.evaluate(np.array(X_test), np.array(y_test))
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Step 11: Function to predict sentiment from new text input
def predict_sentiment(text):
    # Tokenize and pad the input text
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=200)  # Ensure the padding matches the training
    prediction = model.predict(padded)[0][0]
    if prediction >= 0.5:
        return "Positive"
    else:
        return "Negative"

# Step 12: Get user input and predict sentiment
while True:
    user_input = input("Enter a text to analyze sentiment (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    print("Sentiment:", predict_sentiment(user_input))


Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 335ms/step - accuracy: 0.5248 - loss: 0.6926 - val_accuracy: 0.5224 - val_loss: 0.6915
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.5081 - loss: 0.6957 - val_accuracy: 0.5224 - val_loss: 0.6907
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 544ms/step - accuracy: 0.5295 - loss: 0.6917 - val_accuracy: 0.5522 - val_loss: 0.6881
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 662ms/step - accuracy: 0.5292 - loss: 0.6914 - val_accuracy: 0.5299 - val_loss: 0.6871
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 655ms/step - accuracy: 0.5697 - loss: 0.6885 - val_accuracy: 0.5746 - val_loss: 0.6834
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 656ms/step - accuracy: 0.5449 - loss: 0.6871 - val_accuracy: 0.5299 - val_loss: 0.6821
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 635ms/step - accuracy: 0.5366 - loss: 0.6869 - val_accuracy: 0.5299 - val_loss: 0.6824
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 655ms/step - accuracy: 0.5474 - loss: 0.6875 - val_accura